## Author: Gabriel Vance
## Class: DSC630 - Predictive Analytics
## Professor: Professor Andrew Hua
## Date: October 27, 2025

# Purpose: This notebook performs the tasks outlined in 10.2 Assignment: Recommender Systems. 

Using the small MovieLens data set, create a recommender system that allows users to input a movie they like (in the data set) and recommends ten other movies for them to watch. In your write-up, clearly explain the recommender system process and all steps performed. If you are using a method found online, be sure to reference the source.



# Movie Recommender System Implementation

## Overview
This project implements a **collaborative filtering** movie recommender system using the MovieLens dataset. The system uses **item-based collaborative filtering** with **cosine similarity** to recommend movies to users based on their preferences.

## Methodology
The recommender system works by:
1. **Data Loading & Preprocessing**: Loading movie and rating data, creating user-item matrices
2. **Similarity Calculation**: Computing cosine similarity between movies based on user ratings
3. **Recommendation Generation**: Finding movies most similar to a user's input movie
4. **Filtering & Ranking**: Presenting the top 10 most similar movies

## Data Sources
- **Movies Dataset**: Contains movie information (ID, title, genres)
- **Ratings Dataset**: Contains user ratings (userId, movieId, rating, timestamp)

**Reference**: This implementation follows collaborative filtering principles from Ricci, F., Rokach, L., & Shapira, B. (2015). *Recommender Systems Handbook*. Springer.

In [3]:
# Import required libraries
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
import warnings
warnings.filterwarnings('ignore')


In [4]:
# Step 1: Load the MovieLens dataset
print("Loading MovieLens dataset...")

# Load movies data
# movies = pd.read_csv('/workspaces/data_science_classes_windows/fall2025/dsc630_data_prediction/week10/full_data/movies.csv')
movies = pd.read_csv('full_data/movies.csv')

print(f"Movies dataset shape: {movies.shape}")
print(f"Sample movies:")
print(movies.head())

# Load ratings data (using sample.csv as it's a smaller subset)
# ratings = pd.read_csv('/workspaces/data_science_classes_windows/fall2025/dsc630_data_prediction/week10/full_data/sample.csv')
ratings = pd.read_csv('full_data/ratings.csv')

print(f"\nRatings dataset shape: {ratings.shape}")
print(f"Sample ratings:")
print(ratings.head())

print(f"\nUnique users: {ratings['userId'].nunique()}")
print(f"Unique movies: {ratings['movieId'].nunique()}")
print(f"Rating range: {ratings['rating'].min()} - {ratings['rating'].max()}")

Loading MovieLens dataset...
Movies dataset shape: (87585, 3)
Sample movies:
   movieId                               title  \
0        1                    Toy Story (1995)   
1        2                      Jumanji (1995)   
2        3             Grumpier Old Men (1995)   
3        4            Waiting to Exhale (1995)   
4        5  Father of the Bride Part II (1995)   

                                        genres  
0  Adventure|Animation|Children|Comedy|Fantasy  
1                   Adventure|Children|Fantasy  
2                               Comedy|Romance  
3                         Comedy|Drama|Romance  
4                                       Comedy  

Ratings dataset shape: (32000204, 4)
Sample ratings:
   userId  movieId  rating  timestamp
0       1       17     4.0  944249077
1       1       25     1.0  944250228
2       1       29     2.0  943230976
3       1       30     5.0  944249077
4       1       32     5.0  943228858

Unique users: 200948
Unique movies: 84432
Rat

In [ ]:
# Step 2: Data preprocessing and exploration
print("Data Preprocessing...")

# Merge movies and ratings data
movie_ratings = pd.merge(ratings, movies, on='movieId', how='inner')
print(f"Combined dataset shape: {movie_ratings.shape}")

# Check for movies with sufficient ratings (at least 10 ratings for better similarity calculation)
movie_rating_counts = movie_ratings.groupby('movieId').size()
popular_movies = movie_rating_counts[movie_rating_counts >= 10].index
print(f"Movies with at least 10 ratings: {len(popular_movies)}")

# Filter to only include popular movies
filtered_ratings = movie_ratings[movie_ratings['movieId'].isin(popular_movies)]
print(f"Filtered dataset shape: {filtered_ratings.shape}")

# Display most rated movies
top_movies = filtered_ratings.groupby('title').size().sort_values(ascending=False).head(10)
print(f"\nTop 10 most rated movies:")
print(top_movies)

Data Preprocessing...
Combined dataset shape: (32000204, 6)
Movies with at least 10 ratings: 31961
Filtered dataset shape: (31842705, 6)

Top 10 most rated movies:
title
Shawshank Redemption, The (1994)                             102929
Forrest Gump (1994)                                          100296
Pulp Fiction (1994)                                           98409
Matrix, The (1999)                                            93808
Silence of the Lambs, The (1991)                              90330
Star Wars: Episode IV - A New Hope (1977)                     85010
Fight Club (1999)                                             77332
Jurassic Park (1993)                                          75233
Schindler's List (1993)                                       73849
Lord of the Rings: The Fellowship of the Ring, The (2001)     73122
dtype: int64


: 

In [ ]:
# Step 3: Create User-Item Matrix
print("Creating user-item matrix...")

# Create pivot table (user-item matrix) with ratings
user_item_matrix = filtered_ratings.pivot_table(
    index='userId', 
    columns='movieId', 
    values='rating',
    fill_value=0
)

print(f"User-item matrix shape: {user_item_matrix.shape}")
print(f"Sparsity: {(user_item_matrix == 0).sum().sum() / (user_item_matrix.shape[0] * user_item_matrix.shape[1]):.4f}")

# Create movie-user matrix (transpose for item-based filtering)
movie_user_matrix = user_item_matrix.T
print(f"Movie-user matrix shape: {movie_user_matrix.shape}")

# Display sample of the matrix
print("\nSample of user-item matrix:")
print(user_item_matrix.iloc[:5, :5])

Creating user-item matrix...


In [ ]:
# Step 4: Calculate Movie Similarity Matrix using Cosine Similarity
print("Calculating movie similarity matrix...")

# Calculate cosine similarity between movies
# We use the movie-user matrix where each row represents a movie and columns are users
movie_similarity = cosine_similarity(movie_user_matrix)

# Convert to DataFrame for easier handling
movie_similarity_df = pd.DataFrame(
    movie_similarity, 
    index=movie_user_matrix.index, 
    columns=movie_user_matrix.index
)

print(f"Movie similarity matrix shape: {movie_similarity_df.shape}")

# Display sample similarities for first movie
sample_movie_id = movie_similarity_df.index[0]
print(f"\nSimilarity scores for movie ID {sample_movie_id}:")
print(movie_similarity_df.loc[sample_movie_id].sort_values(ascending=False).head(6))

Calculating movie similarity matrix...
Movie similarity matrix shape: (767, 767)

Similarity scores for movie ID 1:
movieId
1       1.000000
3114    0.696673
260     0.629833
1580    0.607824
6377    0.598527
2571    0.596779
Name: 1, dtype: float64
Movie similarity matrix shape: (767, 767)

Similarity scores for movie ID 1:
movieId
1       1.000000
3114    0.696673
260     0.629833
1580    0.607824
6377    0.598527
2571    0.596779
Name: 1, dtype: float64


In [ ]:
# Step 5: Create Movie Recommendation Function
def get_movie_recommendations(movie_title, n_recommendations=10):
    """
    Recommends movies based on similarity to the input movie.
    
    Parameters:
    movie_title (str): The title of the movie to base recommendations on
    n_recommendations (int): Number of recommendations to return
    
    Returns:
    DataFrame: Top N recommended movies with similarity scores
    """
    
    # Find the movie ID for the given title
    movie_match = movies[movies['title'].str.contains(movie_title, case=False, na=False)]
    
    if movie_match.empty:
        print(f"Movie '{movie_title}' not found in the dataset.")
        print("Available movies (sample):")
        print(movies['title'].head(10).tolist())
        return None
    
    # If multiple matches, take the first one
    if len(movie_match) > 1:
        print(f"Multiple movies found with '{movie_title}'. Using: {movie_match.iloc[0]['title']}")
    
    movie_id = movie_match.iloc[0]['movieId']
    movie_title_full = movie_match.iloc[0]['title']
    
    # Check if movie is in our similarity matrix
    if movie_id not in movie_similarity_df.index:
        print(f"Movie '{movie_title_full}' doesn't have enough ratings for recommendations.")
        return None
    
    # Get similarity scores for this movie
    similarity_scores = movie_similarity_df.loc[movie_id]
    
    # Sort by similarity (excluding the movie itself)
    similar_movies = similarity_scores.drop(movie_id).sort_values(ascending=False)
    
    # Get top N recommendations
    top_recommendations = similar_movies.head(n_recommendations)
    
    # Create recommendations dataframe with movie details
    recommendations = []
    for movie_id_rec, similarity_score in top_recommendations.items():
        movie_info = movies[movies['movieId'] == movie_id_rec].iloc[0]
        recommendations.append({
            'movieId': movie_id_rec,
            'title': movie_info['title'],
            'genres': movie_info['genres'],
            'similarity_score': similarity_score
        })
    
    recommendations_df = pd.DataFrame(recommendations)
    
    print(f"\nTop {n_recommendations} recommendations for '{movie_title_full}':")
    print("=" * 70)
    
    for i, row in recommendations_df.iterrows():
        print(f"{i+1:2d}. {row['title']}")
        print(f"    Genres: {row['genres']}")
        print(f"    Similarity Score: {row['similarity_score']:.4f}")
        print()
    
    return recommendations_df

print("Movie recommendation function created successfully!")

Movie recommendation function created successfully!


In [ ]:
# Step 6: Test the Recommender System with Popular Movies
print("Testing the Movie Recommender System...")
print("\n" + "="*80)

# Test with a popular movie
test_movie = "Toy Story"
recommendations = get_movie_recommendations(test_movie)

print("\n" + "="*80)

# Test with another popular movie
test_movie2 = "Jurassic Park"
recommendations2 = get_movie_recommendations(test_movie2)

Testing the Movie Recommender System...

Multiple movies found with 'Toy Story'. Using: Toy Story (1995)

Top 10 recommendations for 'Toy Story (1995)':
 1. Toy Story 2 (1999)
    Genres: Adventure|Animation|Children|Comedy|Fantasy
    Similarity Score: 0.6967

 2. Star Wars: Episode IV - A New Hope (1977)
    Genres: Action|Adventure|Sci-Fi
    Similarity Score: 0.6298

 3. Men in Black (a.k.a. MIB) (1997)
    Genres: Action|Comedy|Sci-Fi
    Similarity Score: 0.6078

 4. Finding Nemo (2003)
    Genres: Adventure|Animation|Children|Comedy
    Similarity Score: 0.5985

 5. Matrix, The (1999)
    Genres: Action|Sci-Fi|Thriller
    Similarity Score: 0.5968

 6. Usual Suspects, The (1995)
    Genres: Crime|Mystery|Thriller
    Similarity Score: 0.5886

 7. Incredibles, The (2004)
    Genres: Action|Adventure|Animation|Children|Comedy
    Similarity Score: 0.5879

 8. Jurassic Park (1993)
    Genres: Action|Adventure|Sci-Fi|Thriller
    Similarity Score: 0.5874

 9. Star Wars: Episode VI -

In [ ]:
# Step 7: Interactive Recommender System
def search_movies(search_term):
    """Helper function to search for movies by title"""
    matches = movies[movies['title'].str.contains(search_term, case=False, na=False)]
    if len(matches) > 0:
        print(f"Movies matching '{search_term}':")
        for i, (idx, movie) in enumerate(matches.head(10).iterrows()):
            print(f"{i+1:2d}. {movie['title']} ({movie['genres']})")
    else:
        print(f"No movies found matching '{search_term}'")
    return matches

def interactive_recommender():
    """Interactive movie recommender function"""
    print("🎬 Welcome to the Movie Recommender System!")
    print("=" * 50)
    print("You can:")
    print("1. Search for movies: search_movies('movie name')")
    print("2. Get recommendations: get_movie_recommendations('movie name')")
    print("3. Example: get_movie_recommendations('Star Wars')")
    print("=" * 50)

# Run the interactive setup
interactive_recommender()

In [ ]:
# Step 8: System Analysis and Evaluation
print("System Analysis and Performance Evaluation")
print("=" * 50)

# Analyze the dataset coverage
print(f"Total movies in dataset: {len(movies)}")
print(f"Movies with ratings: {len(movie_user_matrix)}")
print(f"Coverage: {len(movie_user_matrix) / len(movies) * 100:.2f}%")

# Most popular genres analysis
all_genres = []
for genres in movies['genres']:
    if pd.notna(genres):
        all_genres.extend(genres.split('|'))

genre_counts = pd.Series(all_genres).value_counts().head(10)
print(f"\nTop 10 Genres in Dataset:")
print(genre_counts)

# Test with different movie genres
print(f"\n" + "="*60)
print("Testing recommendations across different genres:")

# Horror movie test
horror_test = "Halloween"
print(f"\n--- Testing Horror Movie: {horror_test} ---")
horror_recs = get_movie_recommendations(horror_test)

# Comedy movie test  
comedy_test = "Dumb and Dumber"
print(f"\n--- Testing Comedy Movie: {comedy_test} ---")
comedy_recs = get_movie_recommendations(comedy_test)

# Recommender System Process Explanation

## 1. Data Loading and Preprocessing
- **Movies Dataset**: Contains 87,587 movies with titles and genres
- **Ratings Dataset**: Sample contains 29,351 ratings from users
- **Data Integration**: Merged ratings with movie information to create comprehensive dataset

## 2. Data Filtering and Quality Control
- **Minimum Rating Threshold**: Only included movies with at least 10 ratings to ensure statistical significance
- **Popular Movies**: Filtered to movies that have sufficient user interaction
- **Matrix Creation**: Built user-item matrix with users as rows and movies as columns

## 3. Item-Based Collaborative Filtering
- **Approach**: Used item-based collaborative filtering rather than user-based
- **Rationale**: Movies have more stable characteristics than user preferences
- **Similarity Metric**: Cosine similarity measures the angle between rating vectors

## 4. Cosine Similarity Calculation
The cosine similarity between two movies A and B is calculated as:

$$\text{similarity}(A,B) = \frac{A \cdot B}{||A|| \times ||B||} = \frac{\sum_{i=1}^{n} A_i \times B_i}{\sqrt{\sum_{i=1}^{n} A_i^2} \times \sqrt{\sum_{i=1}^{n} B_i^2}}$$

Where:
- $A$ and $B$ are rating vectors for movies A and B
- $n$ is the number of users who rated both movies
- Values range from -1 (opposite) to 1 (identical)

## 5. Recommendation Generation
- **Input**: User provides a movie title they enjoyed
- **Process**: System finds the movie in the dataset and retrieves its similarity scores
- **Ranking**: Sorts movies by similarity score (highest first)
- **Output**: Returns top 10 most similar movies with titles, genres, and similarity scores

## System Strengths
- **Cold Start Handling**: Works well for movies with sufficient ratings
- **Scalability**: Efficient for item-based recommendations
- **Interpretability**: Clear similarity scores explain recommendations
- **Genre Awareness**: Recommendations often match or complement genres

## Limitations
- **Data Sparsity**: Limited by movies with few ratings
- **Popularity Bias**: May favor popular movies over niche preferences
- **Static Nature**: Doesn't adapt to changing user preferences over time

## Technical Implementation
- **Libraries**: Used pandas for data manipulation, scikit-learn for similarity calculations
- **Memory Efficiency**: Optimized matrix operations for large datasets
- **Error Handling**: Includes robust search and matching capabilities